In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE orders_bronze AS
SELECT * FROM STREAM READ_FILES(
'/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/orders/',
format => 'json'
);

message
"This Delta Live Tables query is syntactically valid, but you must create a pipeline in order to define and populate your table."


In [0]:
%sql
CREATE LIVE TABLE restaurants_bronze AS
SELECT * FROM READ_FILES(
'/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/dim_restaurants.csv',
format => 'csv',
inferSchema => true,
header => true
);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5510021527285778>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE LIVE TABLE restaurants_bronze AS\nSELECT * FROM READ_FILES(\n'/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/dim_restaurants.csv',\nformat => 'csv',\ninferSchema => true,\nheader => true\n);\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, Fals

In [0]:
%sql
CREATE LIVE TABLE items_bronze AS

SELECT *
FROM READ_FILES(
    '/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/dim_menu_items.csv',
    format => 'csv',
    inferSchema => true,
    header => true
);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6106981062221863>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE LIVE TABLE items_bronze AS\n\nSELECT *\nFROM READ_FILES(\n    '/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/dim_menu_items.csv',\n    format => 'csv',\n    inferSchema => true,\n    header => true\n);\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SI

In [0]:
CREATE LIVE TABLE agents_bronze AS

SELECT *
FROM READ_FILES(
    '/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/dim_delivery_agents.csv',
    format => 'csv',
    inferSchema => true,
    header => true
);

In [0]:
CREATE LIVE TABLE locations_bronze AS

SELECT *
FROM READ_FILES(
    '/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/dim_locations.csv',
    format => 'csv',
    inferSchema => true,
    header => true
);

In [0]:
CREATE LIVE TABLE customers_bronze AS

SELECT *
FROM READ_FILES(
    '/Volumes/manideeppateru_db_workspace_7405606170578073/dine_dash/dinedash/dim_customers.csv',
    format => 'csv',
    inferSchema => true,
    header => true
);

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE cleaned_orders (

    CONSTRAINT valid_order_id
    EXPECT (order_id IS NOT NULL)
    ON VIOLATION DROP ROW,

    CONSTRAINT valid_customer
    EXPECT (customer_id IS NOT NULL)
    ON VIOLATION DROP ROW,

    CONSTRAINT valid_items
    EXPECT (size(items_ordered) > 0)
    ON VIOLATION DROP ROW
)

AS

SELECT
    order_id,
    timestamp AS order_ts,
    customer_id,
    restaurant_id,
    agent_id,
    delivery_location_id,
    items_ordered,
    explode(items_ordered) AS item,
    total_amount,
    tip,
    payment_method,
    order_status

FROM STREAM(Live.orders_bronze);

In [0]:
CREATE OR REFRESH STREAMING TABLE orders_silver
(
    CONSTRAINT valid_restaurant_name
    EXPECT (restaurant_name IS NOT NULL)
    ON VIOLATION DROP ROW,

    CONSTRAINT valid_agent
    EXPECT (agent_name IS NOT NULL)
    ON VIOLATION DROP ROW,

    CONSTRAINT valid_item
    EXPECT (item_id IS NOT NULL)
    ON VIOLATION DROP ROW
)

AS

SELECT
    o.order_id,
    o.order_ts,
    o.customer_id,
    c.name AS customer_name,
    c.email,

    o.restaurant_id,
    r.name AS restaurant_name,
    r.cuisines,
    r.rating AS restaurant_rating,

    o.agent_id,
    a.name AS agent_name,
    a.rating AS agent_rating,

    o.delivery_location_id,
    l.city,

    o.item.item_id AS item_id,
    i.item_name,
    i.category,
    o.item.price AS item_price,
    o.item.quantity AS item_quantity,

    o.total_amount,
    o.tip,
    o.payment_method,
    o.order_status

FROM STREAM(Live.cleaned_orders) o

LEFT JOIN Live.customers_bronze c
    ON o.customer_id = c.customer_id

LEFT JOIN Live.restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id

LEFT JOIN Live.agents_bronze a
    ON o.agent_id = a.agent_id

LEFT JOIN Live.items_bronze i
    ON o.item.item_id = i.item_id

LEFT JOIN Live.locations_bronze l
    ON o.delivery_location_id = l.location_id;

message
"This Delta Live Tables query is syntactically valid, but you must create a pipeline in order to define and populate your table."


In [0]:
CREATE OR REFRESH STREAMING TABLE orders_with_agents AS

SELECT
    o.order_id,
    o.order_ts,
    o.agent_id,
    a.name AS agent_name,
    a.rating AS agent_rating,
    o.tip

FROM STREAM(Live.cleaned_orders) o

LEFT JOIN Live.agents_bronze a
    ON o.agent_id = a.agent_id;

In [0]:
CREATE OR REFRESH STREAMING TABLE orders_with_items AS

SELECT
    o.order_id,
    o.order_ts,
    o.restaurant_id,
    exploded.item_id,
    i.item_name,
    i.category,
    exploded.price,
    exploded.quantity

FROM (
    SELECT
        order_id,
        timestamp AS order_ts,
        restaurant_id,
        explode(items_ordered) AS exploded

    FROM STREAM(Live.orders_bronze)
) o

LEFT JOIN Live.items_bronze i
    ON o.exploded.item_id = i.item_id;

In [0]:
CREATE OR REFRESH STREAMING TABLE orders_with_locations AS

SELECT
    o.order_id,
    o.order_ts,
    o.delivery_location_id,
    l.city,
    l.state,
    l.area

FROM STREAM(Live.cleaned_orders) o

LEFT JOIN Live.locations_bronze l
    ON o.delivery_location_id = l.location_id;

In [0]:
CREATE LIVE TABLE total_orders_per_restaurant AS

SELECT
    restaurant_name,
    COUNT(DISTINCT order_id) AS total_orders

FROM orders_silver

GROUP BY restaurant_name;

In [0]:
CREATE LIVE TABLE top_selling_items AS

SELECT
    item_name,
    SUM(item_quantity) AS total_qty_sold

FROM orders_silver

GROUP BY item_name

ORDER BY total_qty_sold DESC

LIMIT 10;

In [0]:
CREATE LIVE TABLE avg_fees_by_city AS

SELECT
    city,
    AVG(restaurant_rating) AS avg_restaurant_rating,
    AVG(tip) AS avg_tip

FROM orders_silver

GROUP BY city;

In [0]:
CREATE LIVE TABLE agent_order_counts AS

SELECT
    agent_name,
    COUNT(DISTINCT order_id) AS orders_handled,
    AVG(agent_rating) AS avg_rating

FROM orders_silver

GROUP BY agent_name

ORDER BY orders_handled DESC;

In [0]:
CREATE LIVE TABLE daily_revenue_trend AS

SELECT
    DATE(order_ts) AS order_date,
    SUM(total_amount) AS total_revenue

FROM Live.orders_silver

GROUP BY order_date

ORDER BY order_date;

In [0]:
CREATE LIVE TABLE customer_ltv AS

SELECT
    customer_id,
    customer_name,
    SUM(total_amount) AS lifetime_spend

FROM orders_silver

GROUP BY
    customer_id,
    customer_name

ORDER BY lifetime_spend DESC;

In [0]:
CREATE OR REFRESH STREAMING TABLE orders_with_customers AS

SELECT
    o.order_id,
    o.customer_id,
    c.name AS customer_name,
    c.location_id AS customer_location_id,
    o.total_amount,
    o.tip,
    o.payment_method,
    o.order_status

FROM STREAM(Live.cleaned_orders) o

LEFT JOIN Live.customers_bronze c
    ON o.customer_id = c.customer_id;

In [0]:
CREATE LIVE TABLE customer_order_counts AS

SELECT
    customer_name,
    COUNT(*) AS total_orders

FROM orders_with_customers

GROUP BY customer_name;